Transformerベースのモデルを用いた演習を行います。ここでは、日本語の情報を事前学習された軽量モデルを使い、日本語の文章とその感情の情報をファインチューニングした分類モデルを構築します。

以下の事前学習済みのモデルとデータセットを利用します。

- 事前学習済みモデル：distilbert-base-multilingual-cased
- データセット：WRIME

<CPUでの実行について>
Transformerベースのモデルや大規模なデータセットでは、通常GPUを利用します。CPUで実行する場合、計算に非常に時間がかかることが一般的です。

今回の演習では、Transformerモデルを使ったファインチューニングの一連の流れを体験することを優先し、CPUでも実用的に動作するよう、さまざまな調整を行なっています。

- 軽量なモデルの選択
- 訓練データ数の削減
- エポック数を少なくする

そのため、モデルの精度（正解率）は高くありません。ご了承ください。

In [ ]:
import sys
# Python 3.11 の環境に直接インストールを強制(エラーが出ないようにAIに教えてもらった)
!{sys.executable} -m pip install "transformers<5.0.0" "torch>=2.4" tf-keras tensorflow


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
#エラーが出ないようにAIに教えてもらったimport。現在の環境(.venv Python3.11.3を強制使用するため)
import os
import sys

# 1. 環境変数を最優先（インポート前）
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import tensorflow as tf
import tf_keras as keras
import torch
import numpy as np
from transformers import AutoTokenizer, TFAutoModelForSequenceClassification

# 2. 環境の最終確認
print(f"✅ Python Version: {sys.version.split()[0]}")
print(f"✅ PyTorch: {torch.__version__}")
print(f"✅ TensorFlow: {tf.__version__}")

# 3. モデルのロード (WRIME 8クラス分類用)
#model_name = "distilbert-base-multilingual-cased"
#model = TFAutoModelForSequenceClassification.from_pretrained(
#    model_name, 
#    num_labels=8, 
#    from_pt=True
#)

# 4. コンパイル（tf-kerasを使用）
#optimizer = keras.optimizers.Adam(learning_rate=5e-5)
#model.compile(
#    optimizer=optimizer, 
#    loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), 
#    metrics=['accuracy']
#)

# 5. 学習の実行
#print("🚀 ファインチューニングを開始します（5000件）...")
# tokenized_data と emotions が定義されていることを確認してください
#model.fit(
#    x=dict(tokenized_data), 
#    y=emotions, 
#    batch_size=16, 
#    epochs=1
#)

#print("✨ 学習が完了しました！")

✅ Python Version: 3.11.3
✅ PyTorch: 2.10.0
✅ TensorFlow: 2.20.0


In [ ]:
# 必要なライブラリのimport
import numpy as np
import pandas as pd
#import tensorflow as tf  #上記でimport済み
#from transformers import AutoTokenizer, TFAutoModelForSequenceClassification　　#上記でimport済み

In [6]:
# 複数列をまとめる関数を定義
def create_nested_dict(df, prefix):
    emotions = ["Joy", "Sadness", "Anticipation", "Surprise", "Anger", "Fear", "Disgust", "Trust"]
    # 該当する列をフィルタリング
    cols = [col for col in df.columns if col.startswith(prefix)]
    if not cols:
        return pd.Series([None] * len(df))

    # 辞書を作成
    return df[cols].apply(lambda row: {emotion.lower(): row[f"{prefix}{emotion}"] for emotion in emotions if f"{prefix}{emotion}" in row.index}, axis=1)

<データセットの読み込み>
今回は、日本語の感情分析用データセット「WRIME」を利用します。
GitHub上で公開されているTSVファイルをダウンロードし、pandasで前処理 を行います。

なお、前処理では以下を行います。
- 改行の削除や列名の整理
- 複数の列をまとめた新しい列の作成
- 訓練・検証・テストへの分割
CPUでの実行を考慮し、訓練データは一部をサンプルとして使用します。

In [7]:
# 今回利用するデータセットのダウンロード
dataset_df = pd.read_csv("https://github.com/ids-cv/wrime/raw/master/wrime-ver1.tsv", sep="\t")

# データの調整
# 1. Sentence列のデータから改行を削除
dataset_df["Sentence"] = dataset_df["Sentence"].str.replace("\\n", "")

# 2. SentenceとUserIDの列名を変更
dataset_df = dataset_df.rename(columns={"Sentence": "sentence", "UserID": "user_id", "Datetime": "datetime"})

# 3. 複数の列をまとめた新しい列を作成
dataset_df["writer"] = create_nested_dict(dataset_df, "Writer_")
dataset_df["reader1"] = create_nested_dict(dataset_df, "Reader1_")
dataset_df["reader2"] = create_nested_dict(dataset_df, "Reader2_")
dataset_df["reader3"] = create_nested_dict(dataset_df, "Reader3_")
dataset_df["avg_readers"] = create_nested_dict(dataset_df, "Avg. Readers_")

# 4. 訓練データ／検証データ／テストデータに分割し、Train/Dev/Test列を削除
train_df = dataset_df[dataset_df["Train/Dev/Test"] == "train"].reset_index(drop=True)[["sentence", "user_id", "datetime", "writer", "reader1", "reader2", "reader3", "avg_readers"]]
dev_df = dataset_df[dataset_df["Train/Dev/Test"] == "dev"].reset_index(drop=True)[["sentence", "user_id", "datetime", "writer", "reader1", "reader2", "reader3", "avg_readers"]]
test_df = dataset_df[dataset_df["Train/Dev/Test"] == "test"].reset_index(drop=True)[["sentence", "user_id", "datetime", "writer", "reader1", "reader2", "reader3", "avg_readers"]]

# 5. CPUでの実行時間を考慮し、訓練データ数を調整 (例: 5000件)
# もしメモリや時間に余裕があれば、より多くのデータを使用することも可能
num_train_samples = 5000
train_df = train_df[:num_train_samples]

train_df.head()

,sentence,user_id,datetime,writer,reader1,reader2,reader3,avg_readers
0,ぼけっとしてたらこんな時間｡チャリあるから食べにでたいのに…,1,2012/07/31 23:48,"{'joy': 0, 'sadness': 1, 'anticipation': 2, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's..."
1,今日の月も白くて明るい。昨日より雲が少なくてキレイな? と立ち止まる帰り道｡チャリなし生活も...,1,2012/08/02 23:09,"{'joy': 3, 'sadness': 0, 'anticipation': 3, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 2, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's..."
2,早寝するつもりが飲み物がなくなりコンビニへ｡ん､今日、風が涼しいな。,1,2012/08/05 00:50,"{'joy': 1, 'sadness': 1, 'anticipation': 1, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's..."
3,眠い、眠れない。,1,2012/08/08 01:36,"{'joy': 0, 'sadness': 2, 'anticipation': 1, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 1, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 1, 'anticipation': 0, 's..."
4,ただいま? って新体操してるやん!外食する気満々で家に何もないのに!テレビから離れられない…!,1,2012/08/09 22:24,"{'joy': 2, 'sadness': 1, 'anticipation': 3, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 2, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's..."


WRIMEのデータセットは、日本語の文章とその感情の情報が含まれています。
- sentence：文章
- writer：この文章を書いた人の主観による感情の評価
- reader1 〜 reader3：この文章を読んだ3名の客観による感情の評価
- avg_readers：reader1 〜 reader3 の平均値
感情の情報は、Pythonの辞書形式で登録されています。1件取得して、詳細を見てみます。

In [8]:
# trainデータの最初の'avg_readers'を確認
train_df.loc[0, "avg_readers"]

{'joy': 0,
 'sadness': 2,
 'anticipation': 0,
 'surprise': 0,
 'anger': 0,
 'fear': 0,
 'disgust': 0,
 'trust': 0}

8つの感情の要素が0（なし）〜3（強）の4段階で評価されています。

このような辞書形式では扱いづらいため、今回は avg_readers で値の一番大きな感情を採用し、文章（sentence）に紐づけます。さらに、joy や sadness といった文字列を、0, 1, 2, ... といった数値に変換します。

そのため、まずは train_df.loc[0, "avg_readers"] の辞書データのキーをリストに変換します。

In [9]:
# 感情のkeyをリストにする
em_keys = list(train_df.loc[0, "avg_readers"].keys())
print(em_keys)


['joy', 'sadness', 'anticipation', 'surprise', 'anger', 'fear', 'disgust', 'trust']


この em_keys のインデックスを感情の数値として採用します。つまり、joy であれば 0、sadness なら 1 といった具合です。

では、avg_readers の各辞書データで、値の最大値を求め、そのキーを取得します。em_keys.index() を使い、キーを数値に変換し、emotion という列名で train_df に追加します。

In [10]:
# それぞれのDataFrameに'emotion'列を追加し、数値が一番大きいavg_readersのkeyを格納する
train_df["emotion"] = [em_keys.index(max(em, key=em.get)) for em in train_df["avg_readers"]]

train_df.head()

,sentence,user_id,datetime,writer,reader1,reader2,reader3,avg_readers,emotion
0,ぼけっとしてたらこんな時間｡チャリあるから食べにでたいのに…,1,2012/07/31 23:48,"{'joy': 0, 'sadness': 1, 'anticipation': 2, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...",1
1,今日の月も白くて明るい。昨日より雲が少なくてキレイな? と立ち止まる帰り道｡チャリなし生活も...,1,2012/08/02 23:09,"{'joy': 3, 'sadness': 0, 'anticipation': 3, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 2, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's...",3
2,早寝するつもりが飲み物がなくなりコンビニへ｡ん､今日、風が涼しいな。,1,2012/08/05 00:50,"{'joy': 1, 'sadness': 1, 'anticipation': 1, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...",3
3,眠い、眠れない。,1,2012/08/08 01:36,"{'joy': 0, 'sadness': 2, 'anticipation': 1, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 1, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 1, 'anticipation': 0, 's...",1
4,ただいま? って新体操してるやん!外食する気満々で家に何もないのに!テレビから離れられない…!,1,2012/08/09 22:24,"{'joy': 2, 'sadness': 1, 'anticipation': 3, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 2, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's...",0


emotion 列が追加された状態の train_df の内容を確認します。その前に、今回のファインチューニングでは sentence と emotion のみ利用しますので、他の列は削除します。

In [11]:
# sentence列とemotion列のみ残す
train_df = train_df.drop(["user_id", "datetime", "writer", "reader1", "reader2", "reader3", "avg_readers"], axis=1)
train_df.head()

,sentence,emotion
0,ぼけっとしてたらこんな時間｡チャリあるから食べにでたいのに…,1
1,今日の月も白くて明るい。昨日より雲が少なくてキレイな? と立ち止まる帰り道｡チャリなし生活も...,3
2,早寝するつもりが飲み物がなくなりコンビニへ｡ん､今日、風が涼しいな。,3
3,眠い、眠れない。,1
4,ただいま? って新体操してるやん!外食する気満々で家に何もないのに!テレビから離れられない…!,0


train_df のデータについて、emotion の内訳を確認してみます。

In [12]:
# emotionの内訳を確認
train_df.groupby("emotion").count()

,sentence
emotion,
0,1421
1,1100
2,1064
3,887
4,124
5,139
6,231
7,34


<文章データの前処理>
モデルにファインチューニングで学習させるには、文章のデータの前処理が必要です。文章の前処理のために必要なのが「トークナイザー」と呼ばれるものです。トークナイザーによって、文章を単語や文字の塊（トークン）に分割し、モデルが理解できる数値IDに変換します。

Transformerモデルのトークナイザーの機能は、Hugging Faceの Transformers ライブラリの持つ AutoTokenizer で読み込むことができます。今回は distilbert-base-multilingual-cased モデルに対応するトークナイザーを読み込みます。

以上を踏まえ、以下のように記述することで、前処理を実行できます。

In [13]:
# シンボリックリンクの警告を表示させない
import os
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

# sentenceの文章をトークナイザーで前処理する
# 軽量な多言語対応モデルのトークナイザーを指定
tokenizer_name = "distilbert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

tokenized_data = tokenizer(
    train_df["sentence"].to_list(),
    return_tensors="np",  # TensorFlow形式のテンソルで返す
    padding=True,         # 短いシーケンスをパディング
    truncation=True,      # 長いシーケンスを切り捨て
    max_length=512        # モデルが扱える最大長に設定 (多くのモデルで一般的)
)
tokenized_data = dict(tokenized_data)  # 辞書形式に変換しておく
tokenized_data

{'input_ids': array([[  101,  1960, 27849, ...,     0,     0,     0],
        [  101,  2187,  4348, ...,     0,     0,     0],
        [  101,  4352,  3425, ...,     0,     0,     0],
        ...,
        [  101,  5915,  4704, ...,     0,     0,     0],
        [  101,  2531,  2046, ...,     0,     0,     0],
        [  101,   124,  4388, ...,     0,     0,     0]]),
 'attention_mask': array([[1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        ...,
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0]])}

トークナイザーによる前処理で、sentence の文章データが上のように変換されました。

emotion 列についても、ファインチューニングのため、ndarray型に変換しておきます。

In [14]:
# emotion列をndarrayにする
emotions = np.array(train_df["emotion"])

<ファインチューニングしてモデルを構築する>
ここまで準備できたところで、distilbert-base-multilingual-cased のモデルに対してファインチューニングを実施します。ファインチューニングには20～60分かかります。

distilbert-base-multilingual-cased のモデルは、TFAutoModelForSequenceClassification で読み込みます。そうすることで、TensorFlowの keras.Model と同様の形式でモデルを用意できます。また、今回は感情分析を行うため、num_labels には感情の数（8種類）を指定します。

In [ ]:
#import os
#os.environ["TF_USE_LEGACY_KERAS"] = "1"
#import tf_keras
#print(f"✅ Python 3.11 の .venv で成功！ version: {tf_keras.__version__}")

✅ Python 3.11 の .venv で成功！ version: 2.20.1


In [15]:
#Kerasのバックエンドとしてレガシー版を使用するよう指定(TensorFlow 2.16以降でKeras 3.0がデフォルトになったことによる互換性の欠如を回避するため。今回の学習用)
#import os
#os.environ["TF_USE_LEGACY_KERAS"] = "1"
#import tensorflow as tf
#import tf_keras as keras  # レガシー版Kerasとして使用
#from transformers import TFAutoModelForSequenceClassification

# モデルを作成して学習させる
model_name = "distilbert-base-multilingual-cased"
model = TFAutoModelForSequenceClassification.from_pretrained(model_name, num_labels=8, from_pt=True)

# オプティマイザや学習率を設定
# 軽量モデルやCPU実行の場合、学習率やバッチサイズの調整が効果的なことがあります。


#optimizer = tf.keras.optimizers.Adam(learning_rate=5e-5) 
# 【重要】tf.keras だとエラーが出るので、tf.kerasではなく、上で import した keras (tf-keras) を使う
optimizer = keras.optimizers.Adam(learning_rate=5e-5)

# 一般的な学習率
# model.compileのlossは、TFAutoModelForSequenceClassification.from_pretrainedでfrom_pt=TrueとしてPyTorchの重みをロードした場合や、
# カスタム損失関数を使わない場合は、通常モデル内部で処理されるか、フレームワーク標準の損失関数が使われます。
# Hugging FaceのTFモデルでは、loss引数をcompile時に指定せず、fit時に直接ラベルを渡すことで内部損失が使われることが多いです。
# ここでは明示的に`model.hf_compute_loss`を使っていますが、もしこれが原因で別のエラーが出る場合は、
# `loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True)` のような標準的な損失関数を試すことも検討できます。
# ただし、Hugging Faceのドキュメントでは `model.hf_compute_loss` または `loss=model.hf_compute_loss()` の使用が推奨される場合があります。
#model.compile(optimizer=optimizer, loss=model.hf_compute_loss, metrics=['accuracy']) # metricsを追加して精度を監視
model.compile(optimizer=optimizer, loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy']) # metricsを追加して精度を監視



# バッチサイズとエポック数を設定
# CPU実行ではバッチサイズを小さめ (例: 8, 16) に、エポック数も少なめ (例: 1-3) にすると時間短縮になります。
batch_size = 16
epochs = 1 # まずは1エポックで試すなど、調整してください

print("ファインチューニングを開始します。CPUでは時間がかかる場合があります...")
# tokenized_dataは辞書形式である必要があります。{'input_ids': ..., 'attention_mask': ...}
# emotionsはラベルのnumpy配列です。
model.fit(dict(tokenized_data), emotions, batch_size=batch_size, epochs=epochs)
print("ファインチューニングが完了しました。")

Some weights of the PyTorch model were not used when initializing the TF 2.0 model TFDistilBertForSequenceClassification: ['vocab_projector.bias', 'vocab_layer_norm.bias', 'vocab_projector.weight', 'vocab_transform.bias', 'vocab_transform.weight', 'vocab_layer_norm.weight']
- This IS expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model trained on another task or with another architecture (e.g. initializing a TFBertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFDistilBertForSequenceClassification from a PyTorch model that you expect to be exactly identical (e.g. initializing a TFBertForSequenceClassification model from a BertForSequenceClassification model).
Some weights or buffers of the TF 2.0 model TFDistilBertForSequenceClassification were not initialized from the PyTorch model and are newly initialized: ['pre_classifier.weight', 'pre_classifier.bias', 'classifier.weight', 'cla

ファインチューニングを開始します。CPUでは時間がかかる場合があります...
313/313 [==============================] - 932s 3s/step - loss: 1.6755 - accuracy: 0.3128
ファインチューニングが完了しました。


<テストデータを使って評価する>
後は今までのレッスン内容と同様、テストデータを使って predict() を実行し、モデルの性能を評価します。

そのため、テストデータについても、訓練データで行った前処理を実行します。

In [16]:
# testデータについても同様に前処理をしていく
test_df.head()

,sentence,user_id,datetime,writer,reader1,reader2,reader3,avg_readers
0,汗めっちゃかいた(),49,2016/06/18 15:25,"{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 1, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's..."
1,1人だけ春みたいなかっこしててはずい,49,2016/06/18 15:40,"{'joy': 0, 'sadness': 1, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 1, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 1, 'anticipation': 0, 's..."
2,……はあ；；；；,49,2016/06/18 21:36,"{'joy': 2, 'sadness': 0, 'anticipation': 2, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 2, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's..."
3,あぁーテスト勉強(),49,2016/06/19 21:17,"{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 1, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 0, 'sadness': 2, 'anticipation': 0, 's..."
4,ハシが狂っていく様が儚げで見てて辛かったけど、なんか好き…,49,2016/06/19 22:14,"{'joy': 1, 'sadness': 0, 'anticipation': 1, 's...","{'joy': 1, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 2, 'sadness': 2, 'anticipation': 0, 's...","{'joy': 2, 'sadness': 0, 'anticipation': 0, 's...","{'joy': 2, 'sadness': 1, 'anticipation': 0, 's..."


In [17]:
test_df["emotion"] = [em_keys.index(max(em, key=em.get)) for em in test_df["avg_readers"]]
test_df = test_df.drop(["user_id", "datetime", "writer", "reader1", "reader2", "reader3", "avg_readers"], axis=1)
test_df.head()

,sentence,emotion
0,汗めっちゃかいた(),0
1,1人だけ春みたいなかっこしててはずい,1
2,……はあ；；；；,0
3,あぁーテスト勉強(),1
4,ハシが狂っていく様が儚げで見てて辛かったけど、なんか好き…,0


In [18]:
test_tokenized_data = tokenizer(
    test_df["sentence"].to_list(),
    return_tensors="np",
    padding=True,
    truncation=True,
    max_length=512
)
test_tokenized_data = dict(test_tokenized_data)
test_tokenized_data

{'input_ids': array([[  101,  4881,  1965, ...,     0,     0,     0],
        [  101,   122,  2179, ...,     0,     0,     0],
        [  101,   100,   100, ...,     0,     0,     0],
        ...,
        [  101,  2195,  2149, ...,     0,     0,     0],
        [  101,  4422,  1925, ...,     0,     0,     0],
        [  101,  2187, 10906, ...,     0,     0,     0]]),
 'attention_mask': array([[1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        ...,
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0],
        [1, 1, 1, ..., 0, 0, 0]])}

In [19]:
test_emotions = np.array(test_df["emotion"])

In [20]:
# testデータで予測を実施
print("テストデータで予測を開始します...")
predictions = model.predict(test_tokenized_data)
print("予測が完了しました。")

テストデータで予測を開始します...
63/63 [==============================] - 76s 1s/step
予測が完了しました。


ひとまず、predictions をそのまま表示してみます。

In [21]:
# 実行結果をそのまま表示してみる
predictions

TFSequenceClassifierOutput(loss=None, logits=array([[ 1.1675448 ,  1.4892231 ,  0.837582  , ..., -0.94610965,
        -0.4671998 , -2.4485722 ],
       [ 1.5397785 ,  0.7968676 ,  1.9418482 , ..., -1.1803244 ,
        -1.0105081 , -2.7044523 ],
       [ 0.8820175 ,  1.4371687 ,  0.714638  , ..., -0.97586673,
        -0.443075  , -2.3110838 ],
       ...,
       [ 2.143228  ,  0.52620864,  1.7791318 , ..., -1.1503273 ,
        -1.2064695 , -2.4463415 ],
       [ 1.6234714 ,  1.2407916 ,  1.1906112 , ..., -0.993151  ,
        -0.9147288 , -2.5192747 ],
       [ 2.2799702 ,  0.56807655,  1.7799947 , ..., -1.131739  ,
        -1.2352895 , -2.528771  ]], dtype=float32), hidden_states=None, attentions=None)

logits の内容は、各テストデータの8種類の感情について、モデルが評価したそれぞれの感情のスコア（確率に変換する前の値）です。これではわかりづらいので、predictions の logits のデータを np.argmax() で最大値のインデックスを取得します。

In [22]:
# 一番大きい確率のものを予測結果とする形で調整する
# model.predictの出力がTFSequenceClassifierOutputオブジェクトの場合、.logitsでアクセス
pred_emotions = np.argmax(predictions.logits, axis=1)
pred_emotions

array([1, 2, 1, ..., 0, 0, 0])

実際の値（正解値）は test_emotions に格納されています。

In [23]:
# 実際の値を確認
test_emotions

array([0, 1, 0, ..., 2, 6, 2])

これを pred_emotions と比較して、混同行列と正解率を算出します。

In [25]:
from sklearn import metrics

# 混同行列
print("混同行列:")
print(metrics.confusion_matrix(test_emotions, pred_emotions))

混同行列:
[[337 124 124   9   0   0   0   0]
 [106 207  70  10   0   0   0   0]
 [196 115 193   9   0   0   0   0]
 [ 62  51  56  17   0   0   0   0]
 [  5  10   6   1   0   0   0   0]
 [ 31 100  42  13   0   0   0   0]
 [ 24  45  22   8   0   0   0   0]
 [  2   2   3   0   0   0   0   0]]


In [26]:
# 正解率
accuracy = metrics.accuracy_score(test_emotions, pred_emotions)
print(f"正解率: {accuracy:.4f}")

正解率: 0.3770


より精度を上げるには、より多くのデータで学習する（計算資源が必要になります）、エポック数を増やす、ハイパーパラメータ（学習率、バッチサイズなど）を調整する、より高性能なモデルを試すなどの方法が考えられます。いずれもCPU環境では対応できず、GPUが必須となります。